In [ ]:
!pip install gdown

In [ ]:
!gdown --folder https://drive.google.com/drive/folders/1lm5JrRj-v2KjwTnqalWWDggNer5Zkd8Y?usp=sharing

Retrieving folder contents
Processing file 1HgGkoCkgQIHm9RBLHVDk2Ymffj114-51 baseline.ipynb
Processing file 1BluU1LL2Vbp3r7j6xxdePSVrNK1bQ3Ne customers.csv
Processing file 17GPgEUQtEuNJq336zdrRba53he-Pzqm8 geography.csv
Processing file 18TAFcPhelq3zd2hhCeJ2G4TVyikQZhCh inventory.csv
Processing file 1WVZ-qo1PXT0zX5p8cn87sY8jnT7Rs0Ck order_items.csv
Processing file 1bqG6Y1NsbC2JV5ODrUWhvQ4F8LUF0xUu orders.csv
Processing file 1cf5XgkY2Y141meQ0Maqwogviqkqk99ba payments.csv
Processing file 1VSRvIsRgJg5doz--KKpZONlJBAsppdVZ products.csv
Processing file 1Z2LMXiHdbDiQR8TqgTT86QHeamVECNYz promotions.csv
Processing file 1jHTTm7wdkj8fPEwRi5UmmL2qt3xfNPrj returns.csv
Processing file 1c9SvMMJV2BjN2LLl6A-z_R-IYqMHHJCQ reviews.csv
Processing file 1Iv9fhNMifdmx7E_SRzhq6ylut9UuitOx sales.csv
Processing file 1QVlkk8Co4GCwgnK8DJccUcNI9_bIPC2m sample_submission.csv
Processing file 1GC8PSUkoEU-Vm5cVxrge0OZLZmP2xE0d shipments.csv
Processing file 1JJVMpQqSSjzKnzf1RfRRQeRWBrKsn6TY web_traffic.csv
Retrieving f

**Câu 1**

In [ ]:
# Import thư viện
import pandas as pd

# ---------------- Hàm xử lý ----------------
def calculate_inter_order_gap_median(orders_df):
    """
    Tính trung vị số ngày giữa hai lần mua liên tiếp
    cho các khách hàng có nhiều hơn 1 đơn hàng
    """

    # Convert sang datetime
    orders_df['order_date'] = pd.to_datetime(orders_df['order_date'])

    # Sắp xếp theo customer và thời gian
    orders_df = orders_df.sort_values(by=['customer_id', 'order_date'])

    # Tính khoảng cách giữa các đơn hàng liên tiếp
    orders_df['gap_days'] = orders_df.groupby('customer_id')['order_date'].diff().dt.days

    # Lọc khách hàng có nhiều hơn 1 đơn
    valid_customers = orders_df.groupby('customer_id').filter(lambda x: len(x) > 1)

    # Lấy các khoảng gap hợp lệ (không null)
    gaps = valid_customers['gap_days'].dropna()

    # Tính median
    median_gap = gaps.median()

    return median_gap


# ---------------- Example ----------------
# Đọc dữ liệu
orders = pd.read_csv("Data/orders.csv")

print("Đã tải dữ liệu orders.csv")
print("Số dòng dữ liệu:", orders.shape[0])

# Gọi hàm
median_gap = calculate_inter_order_gap_median(orders)

# In kết quả
print("\nKết quả phân tích:")
print("Trung vị số ngày giữa hai lần mua liên tiếp của khách hàng là:", round(median_gap, 2), "ngày")

Đã tải dữ liệu orders.csv
Số dòng dữ liệu: 646945

Kết quả phân tích:
Trung vị số ngày giữa hai lần mua liên tiếp của khách hàng là: 144.0 ngày


**Câu 2**

In [ ]:
# Import thư viện
import pandas as pd

# ---------------- Hàm xử lý ----------------
def find_best_margin_segment(products_df):
    """
    Tìm phân khúc sản phẩm có tỷ suất lợi nhuận gộp trung bình cao nhất
    """

    # Tính gross margin cho từng sản phẩm
    products_df['gross_margin'] = (products_df['price'] - products_df['cogs']) / products_df['price']

    # Tính trung bình theo từng segment
    segment_margin = products_df.groupby('segment')['gross_margin'].mean()

    # Lấy segment có giá trị cao nhất
    best_segment = segment_margin.idxmax()
    best_value = segment_margin.max()

    return best_segment, best_value, segment_margin


# ---------------- Example ----------------
# Đọc dữ liệu (nhớ sửa đúng đường dẫn folder của bạn)
products = pd.read_csv("Data/products.csv")

print("Đã tải dữ liệu products.csv")
print("Số dòng dữ liệu:", products.shape[0])

# Gọi hàm
best_segment, best_value, segment_margin = find_best_margin_segment(products)

# In kết quả
print("\nKết quả phân tích:")
print("Phân khúc có tỷ suất lợi nhuận gộp trung bình cao nhất là:", best_segment)
print("Giá trị trung bình:", round(best_value, 4))

print("\nChi tiết từng phân khúc:")
print(segment_margin)

Đã tải dữ liệu products.csv
Số dòng dữ liệu: 2412

Kết quả phân tích:
Phân khúc có tỷ suất lợi nhuận gộp trung bình cao nhất là: Standard
Giá trị trung bình: 0.3134

Chi tiết từng phân khúc:
segment
Activewear     0.265600
All-weather    0.284176
Balanced       0.258038
Everyday       0.236343
Performance    0.263650
Premium        0.285377
Standard       0.313442
Trendy         0.240758
Name: gross_margin, dtype: float64


**Câu 3**

In [ ]:
# Import thư viện
import pandas as pd

# ---------------- Hàm xử lý ----------------
def find_top_return_reason(returns_df, products_df):
    """
    Tìm lý do trả hàng xuất hiện nhiều nhất
    trong danh mục Streetwear
    """

    # Join 2 bảng theo product_id
    merged_df = returns_df.merge(products_df, on='product_id', how='left')

    # Lọc các sản phẩm thuộc Streetwear
    streetwear_df = merged_df[merged_df['category'] == 'Streetwear']

    # Đếm số lần xuất hiện của return_reason
    reason_counts = streetwear_df['return_reason'].value_counts()

    # Lấy lý do phổ biến nhất
    top_reason = reason_counts.idxmax()
    top_count = reason_counts.max()

    return top_reason, top_count, reason_counts


# ---------------- Example ----------------
# Đọc dữ liệu
returns = pd.read_csv("Data/returns.csv")
products = pd.read_csv("Data/products.csv")

print("Đã tải dữ liệu returns.csv và products.csv")

# Gọi hàm
top_reason, top_count, reason_counts = find_top_return_reason(returns, products)

# In kết quả
print("\nKết quả phân tích:")
print("Lý do trả hàng xuất hiện nhiều nhất là:", top_reason)
print("Số lần xuất hiện:", top_count)

print("\nChi tiết số lần theo từng lý do:")
print(reason_counts)

Đã tải dữ liệu returns.csv và products.csv

Kết quả phân tích:
Lý do trả hàng xuất hiện nhiều nhất là: wrong_size
Số lần xuất hiện: 7626

Chi tiết số lần theo từng lý do:
return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64


**Câu 4**

In [ ]:
# Import thư viện
import pandas as pd

# ---------------- Hàm xử lý ----------------
def find_lowest_bounce_source(web_df):
    """
    Tìm nguồn traffic có bounce_rate trung bình thấp nhất
    """

    # Tính bounce_rate trung bình theo từng nguồn
    bounce_avg = web_df.groupby('traffic_source')['bounce_rate'].mean()

    # Lấy nguồn có giá trị thấp nhất
    best_source = bounce_avg.idxmin()
    best_value = bounce_avg.min()

    return best_source, best_value, bounce_avg


# ---------------- Example ----------------
# Đọc dữ liệu
web_traffic = pd.read_csv("Data/web_traffic.csv")

print("Đã tải dữ liệu web_traffic.csv")
print("Số dòng dữ liệu:", web_traffic.shape[0])

# Gọi hàm
best_source, best_value, bounce_avg = find_lowest_bounce_source(web_traffic)

# In kết quả
print("\nKết quả phân tích:")
print("Nguồn traffic có bounce_rate trung bình thấp nhất là:", best_source)
print("Giá trị trung bình:", round(best_value, 4))

print("\nChi tiết theo từng nguồn:")
print(bounce_avg)

Đã tải dữ liệu web_traffic.csv
Số dòng dữ liệu: 3652

Kết quả phân tích:
Nguồn traffic có bounce_rate trung bình thấp nhất là: email_campaign
Giá trị trung bình: 0.0045

Chi tiết theo từng nguồn:
traffic_source
direct            0.004511
email_campaign    0.004458
organic_search    0.004504
paid_search       0.004478
referral          0.004499
social_media      0.004476
Name: bounce_rate, dtype: float64


**Câu 5**

In [ ]:
# Import thư viện
import pandas as pd

# ---------------- Hàm xử lý ----------------
def calculate_promo_percentage(order_items_df):
    """
    Tính tỷ lệ phần trăm các dòng có áp dụng khuyến mãi
    """

    # Tổng số dòng
    total_rows = len(order_items_df)

    # Số dòng có promo_id không null
    promo_rows = order_items_df['promo_id'].notna().sum()

    # Tính phần trăm
    percentage = (promo_rows / total_rows) * 100

    return percentage, promo_rows, total_rows


# ---------------- Example ----------------
# Đọc dữ liệu
order_items = pd.read_csv("Data/order_items.csv")

print("Đã tải dữ liệu order_items.csv")
print("Số dòng dữ liệu:", order_items.shape[0])

# Gọi hàm
percentage, promo_rows, total_rows = calculate_promo_percentage(order_items)

# In kết quả
print("\nKết quả phân tích:")
print("Số dòng có áp dụng khuyến mãi:", promo_rows)
print("Tổng số dòng:", total_rows)
print("Tỷ lệ phần trăm:", round(percentage, 2), "%")

Đã tải dữ liệu order_items.csv
Số dòng dữ liệu: 714669

Kết quả phân tích:
Số dòng có áp dụng khuyến mãi: 276316
Tổng số dòng: 714669
Tỷ lệ phần trăm: 38.66 %


/tmp/ipykernel_7476/3011634285.py:24: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv("Data/order_items.csv")


**Câu 6**

In [ ]:
# Import thư viện
import pandas as pd

# ---------------- Hàm xử lý ----------------
def find_best_age_group(customers_df, orders_df):
    """
    Tìm nhóm tuổi có số đơn hàng trung bình trên mỗi khách hàng cao nhất
    """

    # Lọc khách có age_group khác null
    customers_df = customers_df[customers_df['age_group'].notna()]

    # Join với orders
    merged_df = orders_df.merge(customers_df, on='customer_id', how='inner')

    # Tổng số đơn theo nhóm tuổi
    total_orders = merged_df.groupby('age_group')['order_id'].count()

    # Số khách hàng theo nhóm tuổi
    total_customers = customers_df.groupby('age_group')['customer_id'].nunique()

    # Tính trung bình
    avg_orders = total_orders / total_customers

    # Lấy nhóm cao nhất
    best_group = avg_orders.idxmax()
    best_value = avg_orders.max()

    return best_group, best_value, avg_orders


# ---------------- Example ----------------
# Đọc dữ liệu
customers = pd.read_csv("Data/customers.csv")
orders = pd.read_csv("Data/orders.csv")

print("Đã tải dữ liệu customers.csv và orders.csv")

# Gọi hàm
best_group, best_value, avg_orders = find_best_age_group(customers, orders)

# In kết quả
print("\nKết quả phân tích:")
print("Nhóm tuổi có số đơn hàng trung bình cao nhất là:", best_group)
print("Giá trị trung bình:", round(best_value, 2))

print("\nChi tiết theo từng nhóm tuổi:")
print(avg_orders)

Đã tải dữ liệu customers.csv và orders.csv

Kết quả phân tích:
Nhóm tuổi có số đơn hàng trung bình cao nhất là: 55+
Giá trị trung bình: 5.41

Chi tiết theo từng nhóm tuổi:
age_group
18-24    5.226656
25-34    5.245226
35-44    5.337343
45-54    5.357241
55+      5.406851
dtype: float64


**Câu 7**

In [ ]:
# Import thư viện
import pandas as pd

# ---------------- Hàm xử lý ----------------
def find_top_region(orders_df, order_items_df, geography_df):
    """
    Tìm region có tổng doanh thu cao nhất
    """

    # Join orders với geography để lấy region
    orders_geo = orders_df.merge(geography_df, on='zip', how='left')

    # Join với order_items
    merged_df = orders_geo.merge(order_items_df, on='order_id', how='inner')

    # Tính doanh thu từng dòng
    merged_df['revenue'] = merged_df['quantity'] * merged_df['unit_price'] - merged_df['discount_amount']

    # Tổng doanh thu theo region
    revenue_by_region = merged_df.groupby('region')['revenue'].sum()

    # Lấy region cao nhất
    top_region = revenue_by_region.idxmax()
    top_value = revenue_by_region.max()

    return top_region, top_value, revenue_by_region


# ---------------- Example ----------------
# Đọc dữ liệu
orders = pd.read_csv("Data/orders.csv")
order_items = pd.read_csv("Data/order_items.csv")
geography = pd.read_csv("Data/geography.csv")

print("Đã tải dữ liệu orders, order_items, geography")

# Gọi hàm
top_region, top_value, revenue_by_region = find_top_region(orders, order_items, geography)

# In kết quả
print("\nKết quả phân tích:")
print("Region có doanh thu cao nhất là:", top_region)
print("Tổng doanh thu:", round(top_value, 2))

print("\nChi tiết theo từng vùng:")
print(revenue_by_region)

/tmp/ipykernel_7476/3179665771.py:32: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv("Data/order_items.csv")


Đã tải dữ liệu orders, order_items, geography

Kết quả phân tích:
Region có doanh thu cao nhất là: East
Tổng doanh thu: 7291150819.12

Chi tiết theo từng vùng:
region
Central    4.719491e+09
East       7.291151e+09
West       3.670227e+09
Name: revenue, dtype: float64


**Câu 8**

In [ ]:
# Import thư viện
import pandas as pd

# ---------------- Hàm xử lý ----------------
def find_top_payment_cancelled(orders_df):
    """
    Tìm phương thức thanh toán được dùng nhiều nhất
    trong các đơn hàng bị huỷ
    """

    # Lọc các đơn bị huỷ
    cancelled_df = orders_df[orders_df['order_status'] == 'cancelled']

    # Đếm số lần xuất hiện của payment_method
    payment_counts = cancelled_df['payment_method'].value_counts()

    # Lấy phương thức phổ biến nhất
    top_method = payment_counts.idxmax()
    top_count = payment_counts.max()

    return top_method, top_count, payment_counts


# ---------------- Example ----------------
# Đọc dữ liệu
orders = pd.read_csv("Data/orders.csv")

print("Đã tải dữ liệu orders.csv")

# Gọi hàm
top_method, top_count, payment_counts = find_top_payment_cancelled(orders)

# In kết quả
print("\nKết quả phân tích:")
print("Phương thức thanh toán được dùng nhiều nhất trong đơn bị huỷ là:", top_method)
print("Số lần:", top_count)

print("\nChi tiết theo từng phương thức:")
print(payment_counts)

Đã tải dữ liệu orders.csv

Kết quả phân tích:
Phương thức thanh toán được dùng nhiều nhất trong đơn bị huỷ là: credit_card
Số lần: 28452

Chi tiết theo từng phương thức:
payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64


**Câu 9**

In [ ]:
# Import thư viện
import pandas as pd

# ---------------- Hàm xử lý ----------------
def find_highest_return_size(order_items_df, returns_df, products_df):
    """
    Tìm kích thước sản phẩm có tỷ lệ trả hàng cao nhất
    """

    # Join order_items với products để lấy size
    oi_products = order_items_df.merge(products_df, on='product_id', how='left')

    # Tổng số order_items theo size
    total_items = oi_products.groupby('size').size()

    # Join returns với products để lấy size
    returns_products = returns_df.merge(products_df, on='product_id', how='left')

    # Tổng số returns theo size
    total_returns = returns_products.groupby('size').size()

    # Tính tỷ lệ return
    return_rate = total_returns / total_items

    # Lấy size cao nhất
    best_size = return_rate.idxmax()
    best_value = return_rate.max()

    return best_size, best_value, return_rate


# ---------------- Example ----------------
# Đọc dữ liệu
order_items = pd.read_csv("Data/order_items.csv")
returns = pd.read_csv("Data/returns.csv")
products = pd.read_csv("Data/products.csv")

print("Đã tải dữ liệu order_items, returns, products")

# Gọi hàm
best_size, best_value, return_rate = find_highest_return_size(order_items, returns, products)

# In kết quả
print("\nKết quả phân tích:")
print("Kích thước có tỷ lệ trả hàng cao nhất là:", best_size)
print("Tỷ lệ:", round(best_value, 4))

print("\nChi tiết theo từng size:")
print(return_rate)

/tmp/ipykernel_7476/1990936991.py:34: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv("Data/order_items.csv")


Đã tải dữ liệu order_items, returns, products

Kết quả phân tích:
Kích thước có tỷ lệ trả hàng cao nhất là: S
Tỷ lệ: 0.0565

Chi tiết theo từng size:
size
L     0.056250
M     0.055660
S     0.056515
XL    0.055200
dtype: float64


**Câu 10**

In [ ]:
# Import thư viện
import pandas as pd

# ---------------- Hàm xử lý ----------------
def find_best_installment_plan(payments_df):
    """
    Tìm kế hoạch trả góp có giá trị thanh toán trung bình cao nhất
    """

    # Tính giá trị thanh toán trung bình theo số kỳ
    avg_payment = payments_df.groupby('installments')['payment_value'].mean()

    # Lấy số kỳ có giá trị cao nhất
    best_plan = avg_payment.idxmax()
    best_value = avg_payment.max()

    return best_plan, best_value, avg_payment


# ---------------- Example ----------------
# Đọc dữ liệu
payments = pd.read_csv("Data/payments.csv")

print("Đã tải dữ liệu payments.csv")

# Gọi hàm
best_plan, best_value, avg_payment = find_best_installment_plan(payments)

# In kết quả
print("\nKết quả phân tích:")
print("Kế hoạch trả góp có giá trị thanh toán trung bình cao nhất là:", best_plan, "kỳ")
print("Giá trị trung bình:", round(best_value, 2))

print("\nChi tiết theo từng kế hoạch:")
print(avg_payment)

Đã tải dữ liệu payments.csv

Kết quả phân tích:
Kế hoạch trả góp có giá trị thanh toán trung bình cao nhất là: 6 kỳ
Giá trị trung bình: 24446.65

Chi tiết theo từng kế hoạch:
installments
1     24113.274166
2       708.473729
3     24399.635486
6     24446.654403
12    24245.772694
Name: payment_value, dtype: float64
